# Recursive tree map

Developing a treemap optimization template. Testing tree: the file/directory tree of `src/vizopt` itself.

In [ ]:
import pathlib

import networkx as nx

from vizopt import introspection

In [ ]:
src_dir = pathlib.Path().resolve().parents[1] / "src" / "vizopt"
src_dir

In [ ]:
file_tree = introspection.build_file_tree(src_dir)
nx.is_arborescence(file_tree), file_tree.number_of_nodes(), file_tree.number_of_edges()

## Baseline: squarified treemap heuristic

`introspection.plot_treemap` already gives a deterministic (non-optimized) layout via `squarify_layout`, using each file's byte size as its weight. Useful as a reference/initializer for the new template.

In [ ]:
introspection.plot_treemap(file_tree, padding=0.025)

## Level 1: raster-based star-vs-star, byte-sum target areas

First recursion level only: apply `RasterStarOptimizer` to the direct children of `src_dir`, with each child's target area set to its subtree byte size (same weights `plot_treemap` above uses). No containment/enclosure yet — these are just mutually-excluding siblings. Zero-size subtrees (e.g. empty `__init__.py`) are dropped, matching `treemap_layout`'s convention.

In [ ]:
import numpy as np

from vizopt.animation import SnapshotCallback
from vizopt.base import OptimConfig
from vizopt.templates.raster_stars import RasterStarOptimizer
from vizopt.templates.star_vs_star import _radius_from_target_area

In [ ]:
sizes = introspection.compute_subtree_sizes(file_tree)
root = pathlib.Path(".")
level1_nodes = [c for c in file_tree.successors(root) if sizes[c] > 0]
level1_names = [n.name for n in level1_nodes]

# Normalize byte sizes to a nicer area scale (mean area 3, matching the
# scale used in raster_based.ipynb's demos) rather than raw byte counts.
raw_sizes = np.array([sizes[n] for n in level1_nodes], dtype=np.float64)
target_areas = (raw_sizes / raw_sizes.mean() * 3.0).tolist()

n_level1 = len(level1_nodes)
list(zip(level1_names, raw_sizes.astype(int), [round(a, 2) for a in target_areas]))

In [ ]:
K = 64
avg_radius = _radius_from_target_area(float(np.mean(target_areas)), K)

# Seed from the squarify layout instead of a uniform ring: with a 67x spread
# between the smallest and largest target area (treemap.py vs. templates), a
# ring sized from the *average* radius starts the small domains overlapped
# by their bigger neighbours once those grow to target size, and the raster
# exclusion gradient is too weak that deep inside a near-total overlap to dig
# them back out. Squarify never starts with an overlap, regardless of size
# spread, and it's already compact, since nothing here currently pulls
# domains together (no attraction term - only exclusion/area/perimeter).
side = float(np.sqrt(sum(target_areas)))
squarify_rect = (-side / 2, -side / 2, side / 2, side / 2)
level1_rects = introspection.treemap_layout(file_tree, root, rect=squarify_rect)
initial_centers = np.array(
    [
        [(level1_rects[n][0] + level1_rects[n][2]) / 2, (level1_rects[n][1] + level1_rects[n][3]) / 2]
        for n in level1_nodes
    ],
    dtype=np.float32,
)

In [ ]:
target_areas

In [ ]:
cb_level1 = SnapshotCallback(every=50)

level1_optimizer = RasterStarOptimizer(
    n_sets=n_level1,
    initial_centers=initial_centers,
    target_areas=target_areas,
    initial_radius=avg_radius,
    grid_resolution=50,
    weight_target_area=20.0,
    weight_area=0.3,
    weight_perimeter=0.3,
    weight_exclusion=10.0,
    weight_smoothness=1.0,
    exclusion_offset=0.05,
    temperature=0.08,
)

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(figsize=(7, 7))
for name, res in zip(level1_names, level1_results):
    cx, cy = res["center"]
    r = res["radii"]
    angs = res["angles"]
    bx = np.append(cx + r * np.cos(angs), cx + r[0] * np.cos(angs[0]))
    by = np.append(cy + r * np.sin(angs), cy + r[0] * np.sin(angs[0]))
    ax.fill(bx, by, alpha=0.25)
    ax.plot(bx, by, lw=1.5)
    ax.text(cx, cy, name, ha="center", va="center", fontsize=8, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.08)
ax.set_title("Level 1: raster star domains sized by subtree byte count")
ax.axis("off")
plt.tight_layout()
plt.show()

## Adding a compactness term

`exclusion_offset` only matters once domains are already touching — but a circle of area *A* is strictly smaller than a rectangle of area *A*, so even squarify's zero-gap tiling leaves every inscribed circle with slack (worst at the rectangle corners). Exclusion is purely repulsive and never engages if nothing is in contact, so it can't pull domains together to close that slack.

`vizopt.components.common.calculate_total_width_penalty_for_circular_layout` already gives a bounding-box compactness penalty (`center ± max_radius` extent). `OptimizationProblem.terms` is a plain list re-read by `optimize()` on every call, so we can append this term to a `RasterStarOptimizer`-built problem directly, without subclassing.

In [ ]:
import jax.numpy as jnp

from vizopt.base import ObjectiveTerm
from vizopt.components.common import calculate_total_width_penalty_for_circular_layout


def _term_compactness(optim_vars, _input_params):
    centers = optim_vars["centers"]
    max_radii = jnp.max(optim_vars["radii"], axis=1)
    return calculate_total_width_penalty_for_circular_layout(centers, max_radii)

In [ ]:
level1_optimizer.problem_ = level1_optimizer._build_problem()
level1_optimizer.problem_.terms.append(
    ObjectiveTerm("compactness", _term_compactness, multiplier=5.0)
)
level1_optimizer.result_ = level1_optimizer.problem_.optimize(
    OptimConfig(n_iters=3000, learning_rate=0.01), callback=cb_level1
)
level1_results = level1_optimizer.sets_

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for name, res in zip(level1_names, level1_results):
    cx, cy = res["center"]
    r = res["radii"]
    angs = res["angles"]
    bx = np.append(cx + r * np.cos(angs), cx + r[0] * np.cos(angs[0]))
    by = np.append(cy + r * np.sin(angs), cy + r[0] * np.sin(angs[0]))
    ax.fill(bx, by, alpha=0.25)
    ax.plot(bx, by, lw=1.5)
    ax.text(cx, cy, name, ha="center", va="center", fontsize=8, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.08)
ax.set_title("Level 1: raster star domains + compactness term")
ax.axis("off")
plt.tight_layout()
plt.show()

## Level 2: recursing into `templates`

Test case: freeze `templates`' optimized boundary from level 1 and lay out its own children inside it.

- **Target areas**: rescaled to `templates`' *achieved* area (from its frozen radii via the star-polygon area formula), not its nominal target — level 1 only approximately hits its target after a finite number of iterations, and compounding that drift with depth would be wrong. A `fill_fraction < 1` leaves slack for packing inefficiency (gaps at corners, exclusion margins) rather than forcing children to claim area they can't actually occupy.
- **Containment**: a new analytic term, `_term_contained_in_frozen_parent` — boundary points of every child must stay inside the frozen parent star. This mirrors `_multi_term_star_enclosure` but the outer side is a constant read from `input_parameters` (the parent's frozen center/radii), not a variable, so it's simpler and cheaper than the general n×n case.
- **Compactness + exclusion** carry over unchanged from level 1.


In [ ]:
def star_polygon_area(radii):
    K = radii.shape[0]
    delta_theta = 2 * np.pi / K
    return 0.5 * np.sin(delta_theta) * np.sum(radii * np.roll(radii, -1))


parent_name = "templates"
parent_idx = level1_names.index(parent_name)
parent_node = level1_nodes[parent_idx]
parent_result = level1_results[parent_idx]
parent_area = star_polygon_area(parent_result["radii"])

level2_nodes = [c for c in file_tree.successors(parent_node) if sizes[c] > 0]
level2_names = [n.name for n in level2_nodes]
n_level2 = len(level2_nodes)

print(f"{parent_name}: achieved area = {parent_area:.3f}")
level2_names

In [ ]:
fill_fraction = 0.85

raw_sizes2 = np.array([sizes[n] for n in level2_nodes], dtype=np.float64)
target_areas2 = (raw_sizes2 / raw_sizes2.sum() * parent_area * fill_fraction).tolist()

parent_center = parent_result["center"]
side2 = float(np.sqrt(parent_area * fill_fraction))
rect2 = (
    parent_center[0] - side2 / 2,
    parent_center[1] - side2 / 2,
    parent_center[0] + side2 / 2,
    parent_center[1] + side2 / 2,
)
level2_rects = introspection.treemap_layout(file_tree, parent_node, rect=rect2)
initial_centers2 = np.array(
    [
        [(level2_rects[n][0] + level2_rects[n][2]) / 2, (level2_rects[n][1] + level2_rects[n][3]) / 2]
        for n in level2_nodes
    ],
    dtype=np.float32,
)

list(zip(level2_names, [round(a, 3) for a in target_areas2]))

In [ ]:
from vizopt.templates.star_vs_star import _dist_and_angle


def _term_contained_in_frozen_parent(optim_vars, input_params):
    """Boundary of every domain must stay inside a fixed (non-optimized) parent star.

    optim_vars keys: "centers" (n_sets, 2), "radii" (n_sets, K)
    input_params keys: "angles" (K,), "parent_center" (2,), "parent_radii" (K,)
        — parent_radii must be interpolated on the same K/angle grid as "angles".
    Optional input_params keys: "containment_offset" (float)
    """
    centers = optim_vars["centers"]
    radii = optim_vars["radii"]
    angles = input_params["angles"]
    parent_center = input_params["parent_center"]
    parent_radii = input_params["parent_radii"]
    n_sets, K = radii.shape

    directions = jnp.stack([jnp.cos(angles), jnp.sin(angles)], axis=-1)  # (K, 2)
    points = centers[:, None, :] + radii[:, :, None] * directions[None, :, :]  # (n_sets, K, 2)

    diff = points - parent_center[None, None, :]  # (n_sets, K, 2)
    dist, alpha = _dist_and_angle(diff)  # (n_sets, K) each

    delta_theta = 2 * jnp.pi / K
    frac_idx = (alpha % (2 * jnp.pi)) / delta_theta
    idx_lo = jnp.floor(frac_idx).astype(jnp.int32) % K
    idx_hi = (idx_lo + 1) % K
    w_hi = frac_idx - jnp.floor(frac_idx)

    r_lo = parent_radii[idx_lo]  # (n_sets, K)
    r_hi = parent_radii[idx_hi]
    r_interp = (1.0 - w_hi) * r_lo + w_hi * r_hi

    offset = input_params.get("containment_offset", 0.0)
    violation = jnp.maximum(0.0, dist - (r_interp - offset))
    return jnp.sum(violation**2)

In [ ]:
cb_level2 = SnapshotCallback(every=50)

level2_optimizer = RasterStarOptimizer(
    n_sets=n_level2,
    initial_centers=initial_centers2,
    target_areas=target_areas2,
    initial_radius=float(np.sqrt(np.mean(target_areas2) / np.pi)),
    grid_resolution=50,
    weight_target_area=20.0,
    weight_area=0.3,
    weight_perimeter=0.3,
    weight_exclusion=10.0,
    weight_smoothness=1.0,
    exclusion_offset=0.05,
    temperature=0.08,
)

level2_optimizer.problem_ = level2_optimizer._build_problem()
level2_optimizer.problem_.input_parameters["parent_center"] = jnp.array(parent_center)
level2_optimizer.problem_.input_parameters["parent_radii"] = jnp.array(parent_result["radii"])
level2_optimizer.problem_.input_parameters["containment_offset"] = 0.02
level2_optimizer.problem_.terms.append(
    ObjectiveTerm("contained_in_parent", _term_contained_in_frozen_parent, multiplier=30.0)
)
level2_optimizer.problem_.terms.append(
    ObjectiveTerm("compactness", _term_compactness, multiplier=5.0)
)

level2_optimizer.result_ = level2_optimizer.problem_.optimize(
    OptimConfig(n_iters=3000, learning_rate=0.01), callback=cb_level2
)
level2_results = level2_optimizer.sets_

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

cx, cy = parent_result["center"]
r, angs = parent_result["radii"], parent_result["angles"]
bx = np.append(cx + r * np.cos(angs), cx + r[0] * np.cos(angs[0]))
by = np.append(cy + r * np.sin(angs), cy + r[0] * np.sin(angs[0]))
ax.plot(bx, by, color="black", lw=2, ls="--", label=parent_name)

for name, res in zip(level2_names, level2_results):
    cx, cy = res["center"]
    r, angs = res["radii"], res["angles"]
    bx = np.append(cx + r * np.cos(angs), cx + r[0] * np.cos(angs[0]))
    by = np.append(cy + r * np.sin(angs), cy + r[0] * np.sin(angs[0]))
    ax.fill(bx, by, alpha=0.35)
    ax.plot(bx, by, lw=1.2)
    ax.text(cx, cy, name, ha="center", va="center", fontsize=7, fontweight="bold")

ax.set_aspect("equal")
ax.autoscale_view()
ax.margins(0.08)
ax.set_title(f"Level 2: children of '{parent_name}' contained in its frozen level-1 boundary")
ax.axis("off")
plt.tight_layout()
plt.show()